# exp007: A vs. B nas 15 categorias do MVTec AD (local)

Experimento intermediário antes do Artigo 1. 740 imagens (50 por categoria, 40 no `toothbrush`), 3 perguntas, Configurações A e B. Roda neste PC, em CPU, com a mesma máquina e os mesmos parâmetros da Fase 4.

Tempo estimado, pela Fase 4 (cerca de 135 s por imagem para as 3 perguntas):

| Etapa | Tempo |
|---|---|
| PatchCore (15 treinos + teste + evidências) | 2 a 4 h |
| Configuração A | cerca de 28 h |
| Configuração B | cerca de 27 h |
| Métricas sem FaithScore | minutos |
| FaithScore (opcional) | várias horas |

Toda etapa retoma de onde parou, então dá para interromper e rodar de novo. Para execuções longas, o terminal com `nohup` é mais seguro que o Jupyter (veja o fim do notebook). Feche outros programas: o SmolVLM usa cerca de 13 GB de RAM.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# O Jupyter abre em notebooks/; os caminhos do projeto são relativos à raiz.
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print(Path.cwd())
assert Path('data/mvtec_ad/mvtec-ad/bottle').is_dir(), 'MVTec AD não encontrado em data/mvtec_ad/mvtec-ad'
assert Path('experiments/exp007_multicategoria/subset.json').exists(), 'rode: python3 -m src.multicategory'

def run(*args):
    # Para no primeiro erro, em vez de seguir com arquivo faltando.
    subprocess.run([sys.executable, '-m', *args], check=True)

## 1. Evidências do PatchCore

Treina um PatchCore por categoria (coreset 0,01, igual ao bottle), roda o teste completo (AUROC de imagem e de pixel) e extrai a evidência das imagens do subconjunto. Salva a cada categoria.

In [ ]:
run('src.configB.exp007_multicategoria.evidences')

## 2. Configuração A (imagem + pergunta)

In [ ]:
run('src.configA.exp007_multicategoria.run')

## 3. Configuração B (imagem + evidência + pergunta)

In [ ]:
run('src.configB.exp007_multicategoria.run')

## 4. Métricas

Acerto por gabarito, fidelidade à evidência, consistência entre perguntas e riqueza saem em minutos. O FaithScore é opcional (secundário e pouco confiável neste domínio, ver exp004); para incluí-lo, use `FAITHSCORE = True`. Ele retoma de onde parou.

In [ ]:
FAITHSCORE = False
flag = ['--faithscore'] if FAITHSCORE else []
run('src.multicategory_metrics', 'score', '--config', 'A', *flag)

In [ ]:
run('src.multicategory_metrics', 'score', '--config', 'B', *flag)

In [ ]:
run('src.multicategory_metrics', 'report')

In [ ]:
import pandas as pd
pd.read_csv('experiments/exp007_multicategoria/results/tabela_por_categoria.csv')

In [ ]:
import json
summary = json.load(open('experiments/exp007_multicategoria/results/resumo_exp007.json'))
print(json.dumps({'overall': summary['overall'], 'paired': summary['paired']}, indent=1)[:6000])

## Alternativa: terminal

Roda em segundo plano e continua se o Jupyter fechar. Na raiz do repo, com a `.venv` ativa:

```bash
nohup sh -c 'python3 -m src.configB.exp007_multicategoria.evidences && python3 -m src.configA.exp007_multicategoria.run && python3 -m src.configB.exp007_multicategoria.run && python3 -m src.multicategory_metrics score --config A && python3 -m src.multicategory_metrics score --config B && python3 -m src.multicategory_metrics report' > experiments/exp007_multicategoria/execucao.log 2>&1 &
tail -f experiments/exp007_multicategoria/execucao.log
```